In [ ]:
# Complete Fruits Classification Notebook - Copy Everything Belo
# ============================================================================
# FRUITS CLASSIFICATION - COMPLETE NOTEBOOK (ALL IN ONE)
# ============================================================================
# Copy this entire code block and paste into ONE Jupyter cell
# Press Shift+Enter to run everything
# ============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from scipy import ndimage
import warnings
import pickle
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!\n")

# ============================================================================
# DATA PROCESSOR CLASS
# ============================================================================
class FruitDataProcessor:
    def __init__(self, img_size=(150, 150)):
        self.img_size = img_size
        self.classes = None

    def load_images_from_folder(self, folder_path):
        images = []
        labels = []

        for fruit_class in os.listdir(folder_path):
            class_path = os.path.join(folder_path, fruit_class)
            if not os.path.isdir(class_path):
                continue

            print(f"Loading {fruit_class}...", end=" ")
            count = 0

            for filename in os.listdir(class_path):
                if filename.endswith(('.jpg', '.png', '.jpeg')):
                    img_path = os.path.join(class_path, filename)
                    try:
                        img = Image.open(img_path).convert('RGB')
                        img = img.resize(self.img_size)
                        img_array = np.array(img) / 255.0
                        images.append(img_array)
                        labels.append(fruit_class)
                        count += 1
                    except:
                        pass

            print(f"✅ {count} images")

        return np.array(images), np.array(labels)

    def prepare_data(self, data_dir, test_size=0.2, val_size=0.2):
        X, y = self.load_images_from_folder(data_dir)

        self.classes = np.unique(y)
        print(f"\n✅ Classes: {list(self.classes)}")
        print(f"✅ Total images: {len(X)}\n")

        y_numeric = np.array([np.where(self.classes == label)[0][0] for label in y])

        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y_numeric, test_size=test_size, random_state=42, stratify=y_numeric
        )

        val_size_adjusted = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=val_size_adjusted, random_state=42, stratify=y_temp
        )

        print(f"Training: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}\n")

        return X_train, X_val, X_test, y_train, y_val, y_test

# ============================================================================
# LOAD DATA - UPDATE THIS PATH TO YOUR DATASET
# ============================================================================
processor = FruitDataProcessor(img_size=(150, 150))
data_dir = 'data/'  # UPDATE THIS PATH

X_train, X_val, X_test, y_train, y_val, y_test = processor.prepare_data(data_dir)

print(f"Data shapes: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}\n")

# ============================================================================
# VISUALIZATION 1: SAMPLE IMAGES
# ============================================================================
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.suptitle('Sample Images from Dataset', fontsize=16, fontweight='bold')

for idx, fruit_class in enumerate(processor.classes[:3]):
    class_indices = np.where(y_train == idx)[0]
    for i in range(5):
        ax = axes[idx, i]
        ax.imshow(X_train[class_indices[i]])
        ax.set_title(fruit_class)
        ax.axis('off')

plt.tight_layout()
os.makedirs('visualizations', exist_ok=True)
plt.savefig('visualizations/01_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/01_samples.png\n")

# ============================================================================
# FEATURE 1: DATA DISTRIBUTION & CLASS BALANCE
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('FEATURE 1: Data Distribution & Class Balance', fontsize=16, fontweight='bold')

# Class distribution
ax1 = axes[0, 0]
class_counts = [np.sum(y_train == idx) for idx in range(len(processor.classes))]
colors = plt.cm.Set3(np.linspace(0, 1, len(processor.classes)))
ax1.bar(processor.classes, class_counts, color=colors)
ax1.set_title('Training Samples per Fruit Class', fontweight='bold')
ax1.set_ylabel('Number of Images')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Data split
ax2 = axes[0, 1]
splits = ['Train', 'Validation', 'Test']
split_counts = [len(X_train), len(X_val), len(X_test)]
colors_split = ['#3498db', '#e74c3c', '#2ecc71']
wedges, texts, autotexts = ax2.pie(split_counts, labels=splits, autopct='%1.1f%%', colors=colors_split)
ax2.set_title('Data Split Distribution', fontweight='bold')
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

# Image info
ax3 = axes[1, 0]
ax3.text(0.5, 0.5,
    f'✅ Image Dimensions: 150 × 150 pixels\n\n'
    f'✅ Total Images: {len(X_train) + len(X_val) + len(X_test)}\n\n'
    f'✅ Fruit Classes: {len(processor.classes)}\n\n'
    f'✅ Normalization: [0, 1] range',
    ha='center', va='center', fontsize=11,
    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
ax3.axis('off')

# Interpretation
ax4 = axes[1, 1]
ax4.text(0.5, 0.5,
    '📊 INTERPRETATION:\n\n'
    'Balanced classes ensure fair training.\n'
    'Consistent image size improves convergence.\n'
    'Normalization to [0,1] helps model learn faster.\n'
    'Good train/val/test split for reliable evaluation.',
    ha='center', va='center', fontsize=10,
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
ax4.axis('off')

plt.tight_layout()
plt.savefig('visualizations/02_feature1_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/02_feature1_distribution.png\n")

# ============================================================================
# FEATURE 2: COLOR CHARACTERISTICS
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('FEATURE 2: Color Characteristics', fontsize=16, fontweight='bold')

# Average colors
ax1 = axes[0, 0]
avg_colors = np.array([np.mean(X_train[y_train == idx], axis=0) for idx in range(len(processor.classes))])
im = ax1.imshow(avg_colors.reshape(len(processor.classes), 1, 3))
ax1.set_yticks(range(len(processor.classes)))
ax1.set_yticklabels(processor.classes)
ax1.set_xticks([])
ax1.set_title('Average Color Profile', fontweight='bold')

# Red channel
ax2 = axes[0, 1]
for idx, fruit in enumerate(processor.classes):
    red_ch = X_train[y_train == idx][:, :, :, 0].flatten()
    ax2.hist(red_ch, alpha=0.5, label=fruit, bins=30)
ax2.set_xlabel('Red Channel (0-1)')
ax2.set_ylabel('Frequency')
ax2.set_title('Red Channel Distribution', fontweight='bold')
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(alpha=0.3)

# Green channel
ax3 = axes[1, 0]
for idx, fruit in enumerate(processor.classes):
    green_ch = X_train[y_train == idx][:, :, :, 1].flatten()
    ax3.hist(green_ch, alpha=0.5, label=fruit, bins=30)
ax3.set_xlabel('Green Channel (0-1)')
ax3.set_ylabel('Frequency')
ax3.set_title('Green Channel Distribution', fontweight='bold')
ax3.legend(loc='upper right', fontsize=8)
ax3.grid(alpha=0.3)

# Interpretation
ax4 = axes[1, 1]
ax4.text(0.5, 0.5,
    '🎨 INTERPRETATION:\n\n'
    'Different fruits have distinct color profiles:\n\n'
    '• RED: High red channel\n'
    '  (apple, cherry, strawberry)\n\n'
    '• YELLOW: High red + green\n'
    '  (banana, mango, pineapple)\n\n'
    '• GREEN: High green channel\n'
    '  (kiwi, guava)\n\n'
    'Color is key for classification!',
    ha='center', va='center', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.7))
ax4.axis('off')

plt.tight_layout()
plt.savefig('visualizations/03_feature2_color.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/03_feature2_color.png\n")

# ============================================================================
# FEATURE 3: TEXTURE & EDGE COMPLEXITY
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('FEATURE 3: Texture & Edge Complexity', fontsize=16, fontweight='bold')

# Texture scores
texture_scores = {}
for idx, fruit in enumerate(processor.classes):
    class_imgs = X_train[y_train == idx][:20]
    edges = [np.mean(ndimage.sobel(np.mean(img, axis=2))) for img in class_imgs]
    texture_scores[fruit] = np.mean(edges)

# Texture complexity
ax1 = axes[0, 0]
fruits_list = list(texture_scores.keys())
scores_list = list(texture_scores.values())
colors = plt.cm.Set3(np.linspace(0, 1, len(fruits_list)))
ax1.barh(fruits_list, scores_list, color=colors)
ax1.set_xlabel('Edge Density (Texture Complexity)')
ax1.set_title('Texture Complexity by Fruit', fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Edge detection example
ax2 = axes[0, 1]
sample_img = X_train[y_train == 0][0]
gray = np.mean(sample_img, axis=2)
edges = ndimage.sobel(gray)
ax2.imshow(edges, cmap='gray')
ax2.set_title(f'Edge Detection Example ({processor.classes[0]})', fontweight='bold')
ax2.axis('off')

# Smoothness vs intensity
ax3 = axes[1, 0]
smoothness_data = []
for idx, fruit in enumerate(processor.classes):
    avg_intensity = np.mean(X_train[y_train == idx])
    smoothness_data.append({'fruit': fruit, 'texture': texture_scores[fruit], 'intensity': avg_intensity})
df_smooth = pd.DataFrame(smoothness_data)
scatter = ax3.scatter(df_smooth['intensity'], df_smooth['texture'], s=300, alpha=0.6, c=range(len(fruits_list)), cmap='Set3')
for _, row in df_smooth.iterrows():
    ax3.annotate(row['fruit'], (row['intensity'], row['texture']), fontsize=9, ha='center', fontweight='bold')
ax3.set_xlabel('Average Pixel Intensity')
ax3.set_ylabel('Texture Complexity')
ax3.set_title('Texture vs Intensity', fontweight='bold')
ax3.grid(alpha=0.3)

# Interpretation
ax4 = axes[1, 1]
ax4.text(0.5, 0.5,
    '🔍 INTERPRETATION:\n\n'
    'Surface texture varies significantly:\n\n'
    '• SMOOTH: Low edge density\n'
    '  (banana, mango, apple)\n\n'
    '• TEXTURED: High edge density\n'
    '  (strawberry, pineapple, coconut)\n\n'
    'Texture provides complementary\n'
    'information to color for robust\n'
    'classification.',
    ha='center', va='center', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
ax4.axis('off')

plt.tight_layout()
plt.savefig('visualizations/04_feature3_texture.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/04_feature3_texture.png\n")

# ============================================================================
# BUILD MODEL
# ============================================================================
print("🚀 Building model with MobileNetV2 transfer learning...\n")

base_model = MobileNetV2(input_shape=(150, 150, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(len(processor.classes), activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model architecture:")
model.summary()

# ============================================================================
# TRAIN MODEL
# ============================================================================
print("\n🚀 Training model...\n")

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=0.00001, verbose=1)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("\n✅ Training complete!\n")

# ============================================================================
# PLOT TRAINING HISTORY
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2, marker='s', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy Over Time')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2, marker='s', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Over Time')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('visualizations/05_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/05_training_history.png\n")

# ============================================================================
# MAKE PREDICTIONS
# ============================================================================
print("📊 Making predictions...\n")

y_val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
y_test_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

# ============================================================================
# CALCULATE METRICS
# ============================================================================
val_acc = accuracy_score(y_val, y_val_pred)
val_prec = precision_score(y_val, y_val_pred, average='weighted', zero_division=0)
val_rec = recall_score(y_val, y_val_pred, average='weighted', zero_division=0)
val_f1 = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)

test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_rec = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print("=" * 60)
print("📊 VALIDATION METRICS")
print("=" * 60)
print(f"Accuracy:  {val_acc:.4f}")
print(f"Precision: {val_prec:.4f}")
print(f"Recall:    {val_rec:.4f}")
print(f"F1 Score:  {val_f1:.4f}")

print("\n" + "=" * 60)
print("📊 TEST METRICS")
print("=" * 60)
print(f"Accuracy:  {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall:    {test_rec:.4f}")
print(f"F1 Score:  {test_f1:.4f}")
print("=" * 60 + "\n")

# ============================================================================
# CONFUSION MATRIX & CLASSIFICATION REPORT
# ============================================================================
cm = confusion_matrix(y_test, y_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=processor.classes, yticklabels=processor.classes,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix (Test Set)', fontweight='bold', fontsize=14)
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

class_report = classification_report(y_test, y_test_pred, target_names=processor.classes, output_dict=True)
report_df = pd.DataFrame(class_report).transpose()

axes[1].axis('off')
table = axes[1].table(cellText=report_df.round(3).values,
                     colLabels=report_df.columns,
                     rowLabels=report_df.index,
                     cellLoc='center',
                     loc='center',
                     bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)
axes[1].set_title('Classification Report', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig('visualizations/06_evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/06_evaluation_metrics.png\n")

# ============================================================================
# SAMPLE PREDICTIONS
# ============================================================================
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
fig.suptitle('Sample Predictions', fontsize=14, fontweight='bold')

test_indices = np.random.choice(len(X_test), 5, replace=False)

for idx, test_idx in enumerate(test_indices):
    img = X_test[test_idx]
    true_label = processor.classes[y_test[test_idx]]
    pred_label = processor.classes[y_test_pred[test_idx]]

    pred = model.predict(np.expand_dims(img, axis=0), verbose=0)[0]
    confidence = pred[y_test_pred[test_idx]]

    axes[idx].imshow(img)
    axes[idx].set_title(f'True: {true_label}\nPred: {pred_label}\nConf: {confidence:.2f}')
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('visualizations/07_sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/07_sample_predictions.png\n")

# ============================================================================
# SAVE MODEL & FILES
# ============================================================================
os.makedirs('models', exist_ok=True)

model_path = 'models/fruit_classifier_model.h5'
model.save(model_path)
print(f"✅ Model saved: {model_path}")

with open('models/fruit_classes.pkl', 'wb') as f:
    pickle.dump(processor.classes, f)
print(f"✅ Classes saved: models/fruit_classes.pkl")

metrics_dict = {
    'val_accuracy': float(val_acc),
    'val_precision': float(val_prec),
    'val_recall': float(val_rec),
    'val_f1': float(val_f1),
    'test_accuracy': float(test_acc),
    'test_precision': float(test_prec),
    'test_recall': float(test_rec),
    'test_f1': float(test_f1)
}

with open('models/metrics.pkl', 'wb') as f:
    pickle.dump(metrics_dict, f)
print(f"✅ Metrics saved: models/metrics.pkl")

print("\n" + "=" * 60)
print("✅ PROJECT COMPLETE!")
print("=" * 60)
print(f"📊 Test Accuracy: {test_acc:.2%}")
print(f"📊 Test F1 Score: {test_f1:.4f}")
print(f"📁 Model: {model_path}")
print(f"📁 Classes: models/fruit_classes.pkl")
print(f"📁 Visualizations: visualizations/")  
print("=" * 60)